# Pipeline

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lana_nlp.preprocessing.data_loader import LyricsDataLoader
from lana_nlp.preprocessing.text_cleaner import TextCleaner
from lana_nlp.features.lyrics_features import LyricsFeatures
from lana_nlp.analysis.lyrics_analyzer import LyricsAnalyzer
from lana_nlp.analysis.readability import ReadabilityAnalyzer
from lana_nlp.analysis.sentiment import SentimentAnalyzer
from lana_nlp.analysis.statistics import StatisticsAnalyzer
from lana_nlp.analysis.vocabulary import VocabularyAnalyzer

## Load the lyrics

In [2]:
loader = LyricsDataLoader(
    Path("../data/raw/lyrics.csv")
)

df = loader.load()

print(df.shape)
df.head()

(154, 5)


,artist,album,song,year,lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ..."
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ..."
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime..."


## Clean the lyrics (basic)

In [3]:
cleaner = TextCleaner()

df["basic_cleaned_lyrics"] = df["lyrics"].apply(cleaner.basic_clean)
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night\nmost of us w...
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done\nand n...
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it im gonna be a s...
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma wouldnt say you were a nice guy\nbut ...
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime\nth...


## NLP cleaned lyrics

In [4]:
df["nlp_cleaned_lyrics"] = df["basic_cleaned_lyrics"].apply(cleaner.nlp_clean)
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics,nlp_cleaned_lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night\nmost of us w...,"[driveby, sunday, night, u, bed, right, turned..."
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done\nand n...,"[another, day, another, day, done, im, gettin,..."
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it im gonna be a s...,"[well, know, know, im, gon, na, star, wont, wo..."
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma wouldnt say you were a nice guy\nbut ...,"[momma, wouldnt, say, nice, guy, youre, 40, jo..."
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime\nth...,"[well, there, somethin, watchin, crime, make, ..."


In [5]:
df.columns

Index(['artist', 'album', 'song', 'year', 'lyrics', 'basic_cleaned_lyrics',
       'nlp_cleaned_lyrics'],
      dtype='str')

## Create Analyzers

In [6]:
analyzer = LyricsAnalyzer(
    df,
    basic_text_column="basic_cleaned_lyrics",
    nlp_text_column="nlp_cleaned_lyrics",
)

df = analyzer.analyze()

In [7]:
df.shape

(154, 12)

In [8]:
df.columns.tolist()

['artist',
 'album',
 'song',
 'year',
 'lyrics',
 'basic_cleaned_lyrics',
 'nlp_cleaned_lyrics',
 'word_count',
 'unique_words',
 'syllable_count',
 'line_count',
 'reading_minutes']

In [9]:
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics,nlp_cleaned_lyrics,word_count,unique_words,syllable_count,line_count,reading_minutes
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night\nmost of us w...,"[driveby, sunday, night, u, bed, right, turned...",248,97,295,37,1.240
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done\nand n...,"[another, day, another, day, done, im, gettin,...",166,64,211,30,0.830
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it im gonna be a s...,"[well, know, know, im, gon, na, star, wont, wo...",84,44,98,15,0.420
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma wouldnt say you were a nice guy\nbut ...,"[momma, wouldnt, say, nice, guy, youre, 40, jo...",277,114,322,37,1.385
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime\nth...,"[well, there, somethin, watchin, crime, make, ...",227,109,268,40,1.135


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   artist                154 non-null    str    
 1   album                 154 non-null    str    
 2   song                  154 non-null    str    
 3   year                  154 non-null    int64  
 4   lyrics                153 non-null    str    
 5   basic_cleaned_lyrics  154 non-null    str    
 6   nlp_cleaned_lyrics    154 non-null    object 
 7   word_count            154 non-null    int64  
 8   unique_words          154 non-null    int64  
 9   syllable_count        154 non-null    int64  
 10  line_count            154 non-null    int64  
 11  reading_minutes       154 non-null    float64
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 14.6+ KB


In [11]:
df.to_csv("../data/processed/lyrics.csv")

In [12]:
album_summary = analyzer.statistics.summary_by_album()
album_summary.head(11)


,songs,avg_words,median_words,min_words,max_words,total_words,avg_reading_minutes
album,,,,,,,
Ultraviolence,16,273.062500,263.5,126,395,4369,1.365312
Lust for Life,16,357.562500,354.0,207,613,5721,1.787813
Did You Know That There's a Tunnel Under Ocean Blvd,16,377.437500,338.5,188,749,6039,1.887188
Blue Banisters,15,279.266667,293.0,0,436,4189,1.396333
Born To Die,15,403.866667,395.0,254,600,6058,2.019333
Sirens,15,208.733333,227.0,84,277,3131,1.043667
Norman Fucking Rockwell!,14,332.285714,313.0,196,478,4652,1.661429
Honeymoon,14,270.285714,270.0,84,411,3784,1.351429
"Lana Del Ray, A.K.A. Lizzy Grant",13,238.230769,213.0,83,413,3097,1.191154


### Do Ultraviolence and Honeymoon have more instrumental sections?

In [18]:
# Calculate words per line column
df["words_per_line"] = (
    df["word_count"] / df["line_count"].replace(0, pd.NA)
)

df.groupby("album")["words_per_line"].mean().sort_values()

album
Paradise                                               5.583152
Lana Del Ray, A.K.A. Lizzy Grant                         5.7268
Lust for Life                                          6.057829
Blue Banisters                                         6.091362
Ultraviolence                                           6.11597
Did You Know That There's a Tunnel Under Ocean Blvd    6.123504
Honeymoon                                              6.136474
Chemtrails Over the Country Club                       6.167297
Norman Fucking Rockwell!                               6.499715
Sirens                                                 6.745097
Born To Die                                            7.159159
Name: words_per_line, dtype: object

In [19]:
df.columns.tolist()

['artist',
 'album',
 'song',
 'year',
 'lyrics',
 'basic_cleaned_lyrics',
 'nlp_cleaned_lyrics',
 'word_count',
 'unique_words',
 'syllable_count',
 'line_count',
 'reading_minutes',
 'words_per_line']